In [1]:
import pandas as pd
import plotly.express as px

# 1. เรียกอ่านข้อมูลจากไฟล์
file_path = "../data/processed/Game_data_clean.xlsx"
df = pd.read_excel(file_path)

# 2. เลือกเฉพาะ 3 ภูมิภาคหลัก แยกตามค่ายเกม (Publisher)
main_regions = ["NA_Sales", "EU_Sales", "JP_Sales"]
df_grouped = df.groupby("Publisher")[main_regions].sum()

# 3. แปลงยอดขายให้เป็น "สัดส่วนเปอร์เซ็นต์ (%)" ของทั้ง 3 ภูมิภาคนี้
df_percentage = df_grouped.div(df_grouped.sum(axis=1), axis=0) * 100
df_percentage.columns = ["NA (%)", "EU (%)", "JP (%)"]

# รีเซ็ตอินเด็กซ์เพื่อให้คอลัมน์ Publisher นำมาวาดกราฟได้
df_heatmap = df_percentage.reset_index()

# แปลงโครงสร้างข้อมูลจากตารางกว้างให้เป็นตารางยาว (Long Format) เพื่อให้ Plotly วาด Heatmap ได้ง่าย
df_melted = df_heatmap.melt(
    id_vars="Publisher", 
    value_vars=["NA (%)", "EU (%)", "JP (%)"],
    var_name="Region", 
    value_name="Percentage"
)

# 4. สร้างแผนภาพความร้อน (Heatmap) ด้วย Plotly
fig = px.density_heatmap(
    df_melted,
    x="Region",
    y="Publisher",
    z="Percentage",
    histfunc="sum",
    color_continuous_scale="YlOrRd",
    text_auto=".1f", # แสดงตัวเลขเปอร์เซ็นต์บนช่อง (ทศนิยม 1 ตำแหน่ง)
    title="Player Behavior: Sales Proportion (%) by Main Regions per Publisher"
)

# ตั้งค่าดีไซน์เพิ่มเติม (ขนาดฟอนต์ และปรับขนาดกราฟ)
fig.update_layout(
    title_font=dict(size=14, family="Arial"),
    xaxis_title="Region",
    yaxis_title="Publisher",
    width=900,
    height=600
)

# ปรับสีแถบด้านข้างให้ระบุชัดเจนว่าเป็นเปอร์เซ็นต์
fig.update_coloraxes(colorbar_title_text="Percentage (%)")

# 5. บันทึกผลลัพธ์เป็นไฟล์ HTML (ไฟล์นี้จะไปอยู่ในโฟลเดอร์เดียวกับโค้ดของคุณ)
output_filename = "game_sales_heatmap.html"
fig.write_html(output_filename)

print(f"สร้างไฟล์กราฟเสร็จเรียบร้อย! สามารถเปิดไฟล์ '{output_filename}' บนเว็บเบราว์เซอร์เพื่อดูผลลัพธ์ได้เลยครับ")

สร้างไฟล์กราฟเสร็จเรียบร้อย! สามารถเปิดไฟล์ 'game_sales_heatmap.html' บนเว็บเบราว์เซอร์เพื่อดูผลลัพธ์ได้เลยครับ


In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# 1. โหลดข้อมูลจากไฟล์
file_path = "../data/processed/Game_data_clean.xlsx"
df = pd.read_excel(file_path)

# แปลงข้อมูลในคอลัมน์ยอดขายให้เป็นตัวเลขเชิงปริมาณ
sales_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Global_Sales"]
for col in sales_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=sales_cols)

# 2. กำหนดตัวแปรต้น (X) และตัวแปรตาม (y)
X = df[["NA_Sales", "EU_Sales", "JP_Sales"]]
y = df["Global_Sales"]

# 3. แบ่งข้อมูลเป็น Train (80%) และ Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. สร้างและฝึกสอนแบบจำลองด้วย Train Set
model = LinearRegression()
model.fit(X_train, y_train)

# 5. พยากรณ์ผลลัพธ์บน Test Set
y_pred_test = model.predict(X_test)

# 6. คำนวณค่า R², RMSE และ MAE
r2 = r2_score(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae = mean_absolute_error(y_test, y_pred_test)

print(f"R-squared (R2): {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

# 7. สร้างกราฟ Interactive ด้วย Plotly
fig = go.Figure()

# เพิ่มจุดข้อมูล (Scatter Plot)
fig.add_trace(
    go.Scatter(
        x=y_test,
        y=y_pred_test,
        mode="markers",
        name="Data Points (Test Set)",
        marker=dict(color="crimson", opacity=0.6, size=7),
        hovertemplate="<b>Actual:</b> %{x:.2f}M<br><b>Predicted:</b> %{y:.2f}M<extra></extra>",
    )
)

# เพิ่มเส้นพยากรณ์สมบูรณ์แบบ (Perfect Fit Line)
ideal_line = np.linspace(y_test.min(), y_test.max(), 100)
fig.add_trace(
    go.Scatter(
        x=ideal_line,
        y=ideal_line,
        mode="lines",
        name="Perfect Fit Line",
        line=dict(color="darkblue", dash="dash", width=2),
    )
)

# ตั้งค่า Layout และชื่อกราฟ
fig.update_layout(
    title={
        "text": f"Fitted Line Plot: Actual vs Predicted Global Sales (Test Set)<br><span style='font-size:12px;'>R2 = {r2:.4f}, RMSE = {rmse:.4f}, MAE = {mae:.4f}</span>",
        "y": 0.95,
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
    },
    xaxis_title="Actual Global Sales (Millions)",
    yaxis_title="Predicted Global Sales (Millions)",
    width=800,
    height=600,
    title_font=dict(size=14, family="Arial"),
    hovermode="closest",
)

# 8. บันทึกผลลัพธ์เป็นไฟล์ HTML ชิ้นที่สาม
output_model_filename = "game_model_fit_plot.html"
fig.write_html(output_model_filename)

print(f"\nสร้างไฟล์กราฟโมเดลเสร็จเรียบร้อย! สามารถเปิดไฟล์ '{output_model_filename}' บนเว็บได้เลยครับ")

R-squared (R2): 0.9927
RMSE: 0.1075
MAE: 0.0359

สร้างไฟล์กราฟโมเดลเสร็จเรียบร้อย! สามารถเปิดไฟล์ 'game_model_fit_plot.html' บนเว็บได้เลยครับ
